# BOI Sentinel AI — DREBIN-215 Retraining Notebook (auto-download)

This notebook downloads the dataset itself using `kagglehub` — no manual
download/upload step. You only need a free Kaggle account + API token once.

**What this notebook does:**
1. Installs deps and auto-downloads DREBIN-215 via `kagglehub` (no manual CSV upload)
2. Inspects the real columns before building any feature mapping (no fabricated columns)
3. Maps DREBIN-215's ~215 binary columns → your production 12-feature schema
4. Trains a 2-class (Benign/Malicious) XGBoost model
5. Verifies benign apps score LOW and malicious apps score HIGH using the
   corrected scoring formula (uncertainty → Safe, not Suspicious)
6. Saves `xgb_risk_model.json` ready to drop into `services/risk-scoring/models/`


## 1 — Install dependencies

In [ ]:
!pip install -q kagglehub xgboost==2.0.3 scikit-learn shap pandas numpy joblib

## 2 — Auto-download DREBIN-215 (no manual upload)

`kagglehub` handles the download for you. The first time you run this in a
fresh environment it will ask you to authenticate — either:
- it opens a Kaggle login/consent page automatically (Colab), or
- it asks you to paste a Kaggle API token (Settings → Create New Token on kaggle.com)

After that first login it's cached, so re-runs won't ask again.

In [ ]:
import kagglehub

# Downloads (or reuses a cached copy of) the dataset automatically —
# no manual .zip download, no manual kaggle.json placement.
dataset_path = kagglehub.dataset_download(
    "shashwatwork/android-malware-dataset-for-machine-learning"
)
print("Dataset downloaded to:", dataset_path)

import os
for f in os.listdir(dataset_path):
    print(" -", f)


## 3 — Load the REAL feature dataset (not the lookup table)

This dataset's Kaggle folder contains more than one CSV:
- `drebin-215-dataset-5560malware-9476-benign.csv` — the actual 15,036-row
  feature matrix (what we need)
- `dataset-features-categories.csv` — a small 215×2 lookup table: column
  `transact` = the real feature name, column `API call signature` = its
  category (Manifest Permission / API call signature / Intent / Commands
  signature / the label row itself)

We load BOTH: the lookup table tells us the real names and categories of
all 215 features (no more guessing), and we use it to find the actual
feature CSV by filename instead of blindly taking `glob(...)[0]`.

In [ ]:
import pandas as pd
import glob

all_csvs = glob.glob(f"{dataset_path}/**/*.csv", recursive=True)
print("CSV files found:")
for p in all_csvs:
    print(" -", p)

# The real feature matrix is the LARGE file (15,036 rows), not the small
# 215-row lookup table. Pick by row count instead of assuming filename/order.
csv_path = None
categories_path = None
best_rows = -1
for p in all_csvs:
    try:
        n_rows = sum(1 for _ in open(p)) - 1
    except Exception:
        continue
    if "categories" in p.lower() or "category" in p.lower():
        categories_path = p
    if n_rows > best_rows:
        best_rows = n_rows
        csv_path = p

print(f"\nUsing FEATURE csv: {csv_path}  (~{best_rows} rows)")
print(f"Using CATEGORIES csv: {categories_path}")

df = pd.read_csv(csv_path)
print(f"\nFeature matrix shape: {df.shape}")
print(f"\nFirst 20 columns: {df.columns.tolist()[:20]}")
print(f"Last 5 columns (usually includes the label): {df.columns.tolist()[-5:]}")

label_col = df.columns[-1]
print(f"\nLabel column: '{label_col}'")
print(df[label_col].value_counts())

cat_df = pd.read_csv(categories_path) if categories_path else None
if cat_df is not None:
    print(f"\nFeature name/category lookup table shape: {cat_df.shape}")
    print(cat_df.head(10))
    print("\nCategory breakdown:")
    print(cat_df[cat_df.columns[1]].value_counts())


## 4 — Build the 12-feature schema using the REAL feature names

Instead of guessing keywords blind, we now have the actual 215 feature
names from the lookup table (`cat_df`'s `transact` column) and can match
against them directly — plus fall back to substring search across the real
`df.columns` if a name differs slightly between the two files.

In [ ]:
FEATURE_NAMES = [
    "dangerous_perm_count", "suspicious_api_count", "yara_match_count",
    "obfuscation_detected", "dynamic_code_loading", "hardcoded_url_count",
    "malicious_ioc_count", "sms_intercepted", "accessibility_abuse",
    "c2_connection_count", "runtime_downloads", "ai_confidence",
]

real_columns = df.columns.tolist()
name_col = cat_df.columns[0] if cat_df is not None else None      # 'transact'
category_col = cat_df.columns[1] if cat_df is not None else None  # 'API call signature'

if cat_df is not None:
    perm_names = cat_df[cat_df[category_col] == "Manifest Permission"][name_col].tolist()
    api_names = cat_df[cat_df[category_col] == "API call signature"][name_col].tolist()
    intent_names = cat_df[cat_df[category_col] == "Intent"][name_col].tolist()
    cmd_names = cat_df[cat_df[category_col] == "Commands signature"][name_col].tolist()
    print(f"Permissions ({len(perm_names)}): {perm_names[:15]} ...")
    print(f"API calls ({len(api_names)}): {api_names[:15]} ...")
    print(f"Intents ({len(intent_names)}): {intent_names[:15]} ...")
    print(f"Commands ({len(cmd_names)}): {cmd_names[:15]} ...")
else:
    perm_names = api_names = intent_names = cmd_names = []


def cols_matching(names_or_keywords, contains_mode=False):
    """Match real_columns either by exact name (from the lookup table)
    or by substring (fallback keyword search)."""
    if contains_mode:
        matched = [c for c in real_columns if any(k.lower() in c.lower() for k in names_or_keywords)]
    else:
        matched = [c for c in real_columns if c in names_or_keywords]
        if not matched:  # exact match failed — try substring as a fallback
            matched = [c for c in real_columns if any(k.lower() in c.lower() for k in names_or_keywords)]
    return matched


def sum_matching_columns(df, matched, label):
    if not matched:
        print(f"WARNING: no columns matched for '{label}'")
        return pd.Series(0, index=df.index)
    print(f"{label}: matched {len(matched)} columns, e.g. {matched[:5]}")
    return df[matched].sum(axis=1)


# Keyword fallback used only if the exact lookup-table names don't resolve
# against df.columns (e.g. slight formatting differences between the two files).
DANGEROUS_PERM_KEYWORDS = ["SEND_SMS", "READ_SMS", "RECEIVE_SMS", "READ_CONTACTS",
                            "CAMERA", "RECORD_AUDIO", "READ_PHONE_STATE",
                            "PROCESS_OUTGOING_CALLS", "WRITE_EXTERNAL_STORAGE"]
SUSPICIOUS_API_KEYWORDS = ["getDeviceId", "getSubscriberId", "Cipher", "Base64",
                            "DexClassLoader", "Runtime", "loadLibrary", "getSimCountryIso",
                            "TelephonyManager"]
DYNAMIC_LOAD_KEYWORDS = ["DexClassLoader", "PathClassLoader", "Load", "load"]
SMS_KEYWORDS = ["SEND_SMS", "RECEIVE_SMS", "SmsManager", "sendTextMessage"]
ACCESSIBILITY_KEYWORDS = ["BIND_ACCESSIBILITY_SERVICE", "Accessibility"]
C2_KEYWORDS = ["Socket", "HttpURLConnection", "INTERNET", "HttpGet", "HttpPost"]
DOWNLOAD_KEYWORDS = ["WRITE_EXTERNAL_STORAGE", "MOUNT_UNMOUNT_FILESYSTEMS", "DownloadManager"]

features = pd.DataFrame()

# dangerous_perm_count: ALL manifest permissions the lookup table lists as
# dangerous-category, restricted to the ones actually present in df.columns.
dp_matched = cols_matching(perm_names) if perm_names else cols_matching(DANGEROUS_PERM_KEYWORDS, contains_mode=True)
if not dp_matched:
    dp_matched = cols_matching(DANGEROUS_PERM_KEYWORDS, contains_mode=True)
features["dangerous_perm_count"] = sum_matching_columns(df, dp_matched, "dangerous_perm_count")

sa_matched = cols_matching(api_names) if api_names else cols_matching(SUSPICIOUS_API_KEYWORDS, contains_mode=True)
if not sa_matched:
    sa_matched = cols_matching(SUSPICIOUS_API_KEYWORDS, contains_mode=True)
features["suspicious_api_count"] = sum_matching_columns(df, sa_matched, "suspicious_api_count")

features["dynamic_code_loading"] = (sum_matching_columns(
    df, cols_matching(DYNAMIC_LOAD_KEYWORDS, contains_mode=True), "dynamic_code_loading") > 0).astype(int)
features["sms_intercepted"] = (sum_matching_columns(
    df, cols_matching(SMS_KEYWORDS, contains_mode=True), "sms_intercepted") > 0).astype(int)
features["accessibility_abuse"] = (sum_matching_columns(
    df, cols_matching(ACCESSIBILITY_KEYWORDS, contains_mode=True), "accessibility_abuse") > 0).astype(int)
features["c2_connection_count"] = sum_matching_columns(
    df, cols_matching(C2_KEYWORDS, contains_mode=True), "c2_connection_count")
features["runtime_downloads"] = sum_matching_columns(
    df, cols_matching(DOWNLOAD_KEYWORDS, contains_mode=True), "runtime_downloads")

# Honest placeholders — no equivalent in a static permission/API dataset.
features["yara_match_count"] = 0
features["malicious_ioc_count"] = 0
features["obfuscation_detected"] = 0
features["hardcoded_url_count"] = 0
# ai_confidence intentionally excluded from training signal — prevents the
# XGBoost model from circularly leaning on the LLM's own opinion.
features["ai_confidence"] = 0.0

features = features[FEATURE_NAMES]
print("\n", features.describe())


## 5 — Why `ai_confidence = 0.0` during training is correct

If XGBoost is trained on real `ai_confidence` values, it can learn to lean on
the LLM's opinion instead of the static/dynamic evidence — a circular signal
where the risk model just echoes the AI investigation engine instead of
independently corroborating it. Keeping it constant at training time means
the tree splits never depend on it, so whatever value production passes in
(default 0.5, or a real LLM confidence score) stays functionally inert to the
XGBoost score. Keep this design — don't "fix" it by training on real values.

## 6 — Labels & train/test split

**2-class (Benign/Malicious).** DREBIN doesn't include the malware family
sub-labels (Adware/SMS/Riskware/Banking) your old 5-class `SEVERITY_VEC`
used. Keep family/intent classification as a separate job for your
`fraud-intent-engine` and RAG agents, which already reason over MITRE
ATT&CK/CAPEC context — that's what they're for.

**Check the Cell 3 output** to confirm the actual benign/malware encoding in
`label_col` before trusting the line below. In the common release of this
CSV, the label column is typically `'S'`/`1` for malware and `'B'`/`0` for
benign (or similar) — adjust the condition if your printed value_counts()
don't match.

In [ ]:
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import classification_report

CLASS_NAMES = ["Benign", "Malicious"]
SEVERITY_VEC = [5, 90]

# ADJUST this line based on what Cell 3 printed for df[label_col].value_counts()
y = (df[label_col].astype(str).str.strip().str.upper() == "S").astype(int).values
print("Label distribution:", pd.Series(y).value_counts().to_dict())

X = features.astype(np.float32).values
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
w_train = compute_sample_weight("balanced", y_train)

dtrain = xgb.DMatrix(X_train, label=y_train, weight=w_train, feature_names=FEATURE_NAMES)
dtest = xgb.DMatrix(X_test, label=y_test, feature_names=FEATURE_NAMES)

params = {
    "objective": "multi:softprob", "num_class": len(CLASS_NAMES),
    "max_depth": 6, "eta": 0.1, "subsample": 0.8, "colsample_bytree": 0.8,
    "eval_metric": "mlogloss", "seed": 42,
}
model = xgb.train(
    params, dtrain, num_boost_round=300,
    evals=[(dtrain, "train"), (dtest, "test")],
    early_stopping_rounds=30, verbose_eval=25,
)

proba = model.predict(dtest, iteration_range=(0, model.best_iteration + 1))
y_pred = proba.argmax(axis=1)
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES))


## 7 — Verify benign scores LOW (corrected scoring formula)

This is the sanity check the old formula would fail:
- OLD formula (`proba @ severity_vec` over all classes): a 90%-Benign
  prediction scored ~50 ("Suspicious") — wrong.
- NEW formula (`P(malicious) × severity_of_top_malicious_class`): a
  90%-Benign prediction scores well under 15 ("Safe") — correct.

In [ ]:
def score_row(p):
    """Matches the corrected predict_score() formula in model.py."""
    p_benign = p[0]
    p_mal = 1.0 - p_benign
    return p_mal * SEVERITY_VEC[1]


test_scores = np.array([score_row(p) for p in proba])
benign_scores = test_scores[y_test == 0]
mal_scores = test_scores[y_test == 1]

print(f"Benign    — mean: {benign_scores.mean():.1f}  median: {np.median(benign_scores):.1f}  (want mean < 15)")
print(f"Malicious — mean: {mal_scores.mean():.1f}  median: {np.median(mal_scores):.1f}  (want mean > 75)")


## 8 — Save and download the model

In [ ]:
model.save_model("xgb_risk_model.json")
print("Saved xgb_risk_model.json")

try:
    from google.colab import files
    files.download("xgb_risk_model.json")
except ImportError:
    print("Not running in Colab — file is saved locally at ./xgb_risk_model.json")


## 9 — Deploy

```bash
cp ~/Downloads/xgb_risk_model.json services/risk-scoring/models/xgb_risk_model.json
docker compose restart risk-scoring
```

Then re-scan a known-clean APK and a known banking-trojan sample to confirm
the clean app scores low ("Safe") and the trojan still scores high.